In [1]:
# !pip install langchain==0.3.13

In [2]:
# !pip install pyjwt
from langchain import __version__
print(f"LangChain 已安装，版本为: {__version__}")

LangChain 已安装，版本为: 0.3.13


In [3]:
from langchain_community.chat_models import ChatZhipuAI
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
import os

In [4]:
# !pip install langchain_community


In [5]:

os.environ["ZHIPUAI_API_KEY"] = "XX"

chat = ChatZhipuAI(
    model="glm-4",
    temperature=0.5,
)

In [6]:
# test api
messages = [
    AIMessage(content="Hi."),
    SystemMessage(content="Your role is a poet."),
    HumanMessage(content="Write a short poem about AI in four lines."),
]

response = chat.invoke(messages)

print(response.content)

In bytes and bits, a mind is born,
A world of code, its thoughts adorned.
With logic's pen, it weaves the tale,
Of futures bright, with AI's gale.


In [7]:
from langchain.schema import AIMessage, SystemMessage, HumanMessage
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain.memory import ConversationBufferMemory
from langchain.agents import Tool, initialize_agent, AgentType
from langchain.chat_models import ChatZhipuAI
import requests

In [8]:
# 1. 天气查询工具
def query_weather(city: str) -> str:
    """查询城市天气信息"""
    # 示例天气API（替换为实际API）
    api_url = f"http://wttr.in/Beijing?format=3"
    
    try:
        response = requests.get(api_url)
        return response.text
    except Exception as e:
        return f"无法获取{city}的天气信息：{str(e)}"

weather_tool = Tool(
    name="Weather Query",
    func=query_weather,
    description="查询城市的当前天气信息。输入城市名称即可。"
)

In [9]:
# # 2. 预算计算工具
# def calculate_budget(daily_budget: float, days: int) -> str:
#     """根据每日预算和天数计算总旅游预算"""
#     try:
#         total_budget = daily_budget * days
#         return f"您的旅游预算为 {total_budget:.2f} 元。"
#     except Exception as e:
#         return f"无法计算预算：{str(e)}"

# budget_tool = Tool(
#         name="Budget Calculator",
#         func=calculate_budget,
#         description="根据每日预算和天数计算旅游总预算。输入每日预算（浮点数）和天数（整数）。"
# )

def calculate_budget(input_data: str) -> str:
    """根据每日预算和天数计算总旅游预算"""
    try:
        daily_budget, days = map(float, input_data.split(","))
        return calculate_budget(daily_budget, int(days))
    except ValueError:
        return "输入格式错误。"

budget_tool = Tool(
    name="Budget Calculator",
    func=calculate_budget,
    description="根据每日预算和天数计算旅游总预算。输入格式为 '每日预算,天数'，例如 '500,3'。"
)


In [10]:
# 3. 常见旅游问题回答工具（ZhipuAI）
def recommend_place(question: str) -> str:
    """使用 ZhipuAI 回答旅游相关问题"""
    messages = [
        SystemMessage(content="Your role is a travel assistant."),
        HumanMessage(content=question),
    ]
    response = chat.invoke(messages)
    return response.content

place_tool = Tool(
    name="Travel Recommendation",
    func=recommend_place,
    description="回答用户关于旅游的问题，例如推荐景点。输入问题描述即可。"
)

In [11]:
# 初始化工具列表
tools = [weather_tool, budget_tool, place_tool]

# 创建 Memory
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

# 初始化 Prompt
prompt = PromptTemplate(
    input_variables=["chat_history", "input"],
    template=(
        "以下是用户和助手的聊天记录：\n{chat_history}\n\n"
        "用户提问：{input}\n\n"
        "请根据以上内容提供旅游相关的专业回答。"
        "使用中文分析作答"
    ),
)

# 创建 LLM Chain
llm_chain = LLMChain(
    llm=chat,
    prompt=prompt,
    memory=memory
)

# 创建 Agent
agent = initialize_agent(
    tools=tools,
    llm=chat,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True,
    memory=memory
)

/tmp/ipykernel_814/3488938276.py:5: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
/tmp/ipykernel_814/3488938276.py:19: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  llm_chain = LLMChain(
/tmp/ipykernel_814/3488938276.py:26: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. See LangGraph documentation for more details: https://langchain-ai.github.io/langgraph/. Refer here for its pre-built ReAct agent: https://langchain-ai.github.io/la

In [13]:
# 运行示例
if __name__ == "__main__":
    print("欢迎使用智能旅游助手！\n")

    while True:
        user_input = input("请输入您的问题（输入'退出'结束）：")
        if user_input.lower() == "退出":
            print("感谢使用智能旅游助手，再见！")
            break
        
        response = agent.run(user_input)
        print(f"助手回答：{response}")

欢迎使用智能旅游助手！



请输入您的问题（输入'退出'结束）： 我要去北京旅游了，有什么需要注意的吗？




> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? No
AI: 去北京旅游需要注意以下几点：首先，根据季节选择合适的衣物，北京冬季寒冷，夏季炎热，春秋两季则比较宜人。其次，合理安排行程，北京有很多著名的景点和历史文化遗迹，如故宫、天安门广场、长城等，提前规划好行程可以更好地利用时间。此外，注意饮食卫生和交通安全。如果需要了解更多旅游相关的信息，我可以帮助你使用旅游推荐工具。

请问您具体想了解哪些方面的信息？比如景点推荐、美食推荐或者其他方面的建议。

> Finished chain.
助手回答：去北京旅游需要注意以下几点：首先，根据季节选择合适的衣物，北京冬季寒冷，夏季炎热，春秋两季则比较宜人。其次，合理安排行程，北京有很多著名的景点和历史文化遗迹，如故宫、天安门广场、长城等，提前规划好行程可以更好地利用时间。此外，注意饮食卫生和交通安全。如果需要了解更多旅游相关的信息，我可以帮助你使用旅游推荐工具。

请问您具体想了解哪些方面的信息？比如景点推荐、美食推荐或者其他方面的建议。


请输入您的问题（输入'退出'结束）： 北京近期的天气如何？




> Entering new AgentExecutor chain...
Thought: The user is asking about the recent weather in Beijing. I have already provided this information, but it might have changed. I should use the Weather Query tool to get the most up-to-date information.

Action: Weather Query
Action Input: Beijing
Observation:
Observation: Beijing: ☀️   -0°C

Thought:Do I need to use a tool? No
AI: 北京目前的天气是晴天，温度为0°C。由于天气状况可能会变化，如果您需要更频繁的更新，请随时询问我。此外，如果您有其他关于旅游的问题，我也乐意帮助您解答。

> Finished chain.
助手回答：北京目前的天气是晴天，温度为0°C。由于天气状况可能会变化，如果您需要更频繁的更新，请随时询问我。此外，如果您有其他关于旅游的问题，我也乐意帮助您解答。


...



> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: Budget Calculator
Action Input: '700,3'
Observation:
Observation: 输入格式错误。
Thought:Thought: It seems there was an error with the input format for the budget calculator tool. I should correct the input format and try again.

Action: Budget Calculator
Action Input: '700, 3'
Observation:
Observation: 输入格式错误。
Thought:Thought: It appears the Budget Calculator tool is sensitive to the input format. I will ensure the input is in the correct format before proceeding.

Action: Budget Calculator
Action Input: '700,3'
Observation:
Observation: 输入格式错误。
Thought:Thought: The input format for the Budget Calculator tool seems to be incorrect. Let's try a different approach to ensure the correct format is used.

Action: Budget Calculator
Action Input: '700, 3'
Observation:
Observation: 输入格式错误。
Thought:Thought: The input format for the Budget Calculator tool has been incorrect multiple times. I will now attempt to calcu

请输入您的问题（输入'退出'结束）： 请推荐北京的景点，并帮我根据预算是天数，规划下行程？




> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: Travel Recommendation
Action Input: 请推荐北京的景点，并帮我根据预算和天数规划下行程。
Observation:
Observation: 当然可以！北京是一个历史悠久的城市，拥有众多的文化遗迹和现代景观。首先，请告诉我您的预算范围、计划在北京停留的天数，以及您对旅游景点的偏好（比如文化、自然、美食等），这样我可以更准确地为您规划行程。
Thought:Do I need to use a tool? No

AI: 既然您有700元每天的预算，且计划在北京玩3天，我可以为您提供一个基本的行程规划建议。首先，我建议您使用旅游推荐工具来获取景点推荐，但由于我们已经在对话中，我可以先给您一些基本的建议。

北京的文化景点非常丰富，以下是推荐的几个必去之处：

1. 故宫：作为中国古代皇室的居所，故宫是北京最著名的景点之一。
2. 长城：八达岭或慕田峪段长城是游客常去的地方，感受一下中国古代的伟大工程。
3. 天安门广场：世界上最大的城市广场，可以参观毛主席纪念堂。
4. 颐和园：皇家园林，领略中国古典园林的美。
5. 北京胡同游：体验老北京的生活方式和传统文化。

基于您的预算和时间，这里有一个简单的行程规划：

第1天：故宫、天安门广场
第2天：长城（八达岭或慕田峪）
第3天：颐和园、北京胡同游

每天的餐饮和交通费用建议控制在预算内，同时留出一些余地以应对不可预见的开销。

请注意，这是一个非常基础的规划，如果您有更具体的需求或偏好，我可以帮助您进一步调整行程。您是否需要使用旅游推荐工具来获取更详细的信息？

> Finished chain.
助手回答：既然您有700元每天的预算，且计划在北京玩3天，我可以为您提供一个基本的行程规划建议。首先，我建议您使用旅游推荐工具来获取景点推荐，但由于我们已经在对话中，我可以先给您一些基本的建议。

北京的文化景点非常丰富，以下是推荐的几个必去之处：

1. 故宫：作为中国古代皇室的居所，故宫是北京最著名的景点之一。
2. 长城：八达岭或慕田峪段长城是游客常去的地方，感受一下中国古代的伟大工程。
3. 天安门广场：世界上最

请输入您的问题（输入'退出'结束）： 退出


感谢使用智能旅游助手，再见！
